In [1]:
# ==========================================
# CELL 1: IMPORT THƯ VIỆN & CỐ ĐỊNH SEED
# ==========================================
import os
import time 
import math
import random
import psutil
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from torch.cuda.amp import autocast, GradScaler
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from thop import profile # pip install thop
from tqdm.auto import tqdm
import gc
import json
import warnings
warnings.filterwarnings('ignore')

# Module Routing (Cần có sẵn các file .py trong cùng thư mục)
from routing_smoe import SMoELayer
from routing_micro import MICROMoELayer
from routing_expert_choice import ExpertChoiceMoELayer
from routing_adaptive import AdaptiveDynamicMoELayer
from routing_deepseek import DeepSeekMoELayer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f" Đang chạy trên thiết bị: {device}")

def set_seed(seed):
    """Cố định seed để đảm bảo Reproducibility cho bài báo"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    
    print(f" Đã thiết lập Seed = {seed}")

 Đang chạy trên thiết bị: cuda


In [2]:
# ==========================================
# CELL 2: CẤU HÌNH HỆ THỐNG
# ==========================================


class Config:
    MODEL_NAME = "vinai/phobert-large"
    TRAIN_CSV = r"D:\my_project\MoE_Adversarial_NLI\train.csv"
    VAL_CSV = r"D:\my_project\MoE_Adversarial_NLI\validation.csv"
    TEST_CSV = r"D:\my_project\MoE_Adversarial_NLI\test.csv"
    MAX_LEN = 256
    NUM_LABELS = 3
    BATCH_SIZE = 16
    LR = 2e-5
    EPOCHS = 50         
    PATIENCE = 3        
    NUM_EXPERTS = 8
    SEEDS = [42] 
    ROUTING_TYPES_TO_TEST = ["smoe", "micro", "expert_choice", "adaptive", "deepseek"]
    ROUTING_THRESHOLDS = 0.5 
    CAPACITY_FACTOR = 1.2 

# ================= QUẢN LÝ THƯ MỤC THỰC NGHIỆM =================
Config.BASE_DIR = "experiments/PhoBERT"  
Config.RESUME_EXPERIMENT = True  # MỚI: Đặt True để chạy tiếp folder đang dang dở, False để tạo mới

os.makedirs(Config.BASE_DIR, exist_ok=True)
existing_exps = [d for d in os.listdir(Config.BASE_DIR) if d.startswith("experiment_")]
exp_nums = [int(d.split("_")[1]) for d in existing_exps if len(d.split("_")) > 1 and d.split("_")[1].isdigit()]

if Config.RESUME_EXPERIMENT and exp_nums:
    next_exp = max(exp_nums)
    print(f"🔄 CHẾ ĐỘ RESUME: Chạy tiếp tục tại phiên thực nghiệm {next_exp}")
else:
    next_exp = max(exp_nums) + 1 if exp_nums else 1
    print(f"📁 CHẾ ĐỘ NEW: Đã tạo phiên thực nghiệm mới experiment_{next_exp}")

Config.EXP_DIR = os.path.join(Config.BASE_DIR, f"experiment_{next_exp}")
os.makedirs(Config.EXP_DIR, exist_ok=True)

Config.CHECKPOINT_DIR = Config.EXP_DIR
Config.TRAIN_LOG_CSV = os.path.join(Config.EXP_DIR, "training_log.csv")
Config.RESULTS_CSV = os.path.join(Config.EXP_DIR, "result.csv")
Config.HYPERPARAMS_JSON = os.path.join(Config.EXP_DIR, "hyperparameters.json")

📁 CHẾ ĐỘ NEW: Đã tạo phiên thực nghiệm mới experiment_1


In [3]:
# ==========================================
# CELL 3: DATASET & DATALOADER (TỐI ƯU HÓA)
# ==========================================
label_map = {'entailment': 0, 'neutral': 1, 'contradiction': 2}
tokenizer = AutoTokenizer.from_pretrained(Config.MODEL_NAME)

class AdversarialNLIDataset(Dataset):
    def __init__(self, df, tokenizer, max_len):
        self.labels = torch.tensor([label_map.get(str(l).strip().lower(), 1) for l in df['label']], dtype=torch.long)
        
        # Tokenize toàn bộ dataset một lần duy nhất vào RAM
        print(f" Đang pre-tokenize {len(df)} mẫu dữ liệu... Vui lòng đợi.")
        self.encodings = tokenizer(
            df['premise'].tolist(), 
            df['hypothesis'].tolist(),
            add_special_tokens=True,
            max_length=max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        print(" Pre-tokenize hoàn tất!")
        
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        # Chỉ trả về tensor đã lưu sẵn
        return {
            'input_ids': self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'labels': self.labels[idx]
        }

# Đọc dữ liệu
df_train = pd.read_csv(Config.TRAIN_CSV).dropna().reset_index(drop=True)
df_val = pd.read_csv(Config.VAL_CSV).dropna().reset_index(drop=True)
df_test = pd.read_csv(Config.TEST_CSV).dropna().reset_index(drop=True)

# Khởi tạo DataLoader với pin_memory=True, BỎ num_workers
train_loader = DataLoader(
    AdversarialNLIDataset(df_train, tokenizer, Config.MAX_LEN), 
    batch_size=Config.BATCH_SIZE, 
    shuffle=True, 
    pin_memory=False
)
val_loader = DataLoader(
    AdversarialNLIDataset(df_val, tokenizer, Config.MAX_LEN), 
    batch_size=Config.BATCH_SIZE, 
    pin_memory=False
)
test_loader = DataLoader(
    AdversarialNLIDataset(df_test, tokenizer, Config.MAX_LEN), 
    batch_size=Config.BATCH_SIZE, 
    pin_memory=False
)

 Đang pre-tokenize 8012 mẫu dữ liệu... Vui lòng đợi.
 Pre-tokenize hoàn tất!
 Đang pre-tokenize 1000 mẫu dữ liệu... Vui lòng đợi.
 Pre-tokenize hoàn tất!
 Đang pre-tokenize 1000 mẫu dữ liệu... Vui lòng đợi.
 Pre-tokenize hoàn tất!


In [4]:
# ==========================================
# CELL 4: ARCHITECTURE & UTILITIES
# ==========================================
class CheckpointManager:
    def __init__(self, model, optimizer, scheduler, scaler, model_name="moe"):
        self.model = model
        self.optimizer = optimizer
        self.scheduler = scheduler
        self.scaler = scaler
        self.model_name = model_name
        self.best_checkpoint_path = os.path.join(Config.CHECKPOINT_DIR, f"{model_name}_best.pt")
        self.last_checkpoint_path = os.path.join(Config.CHECKPOINT_DIR, f"{model_name}_last.pt")
        self.train_log_path = Config.TRAIN_LOG_CSV
        
        if not os.path.exists(self.train_log_path):
            df = pd.DataFrame(columns=["Seed", "Epoch", "Routing", "Val_Acc", "Val_F1", "Val_Runtime_ms", "Val_VRAM_MB", "Val_Entropy", "Val_Expert_Usage"])
            df.to_csv(self.train_log_path, index=False)

    def save_checkpoint(self, epoch, val_f1, val_acc, is_best=False):
        state = {
            'epoch': epoch, 
            'model_state': self.model.state_dict(),
            'optimizer_state': self.optimizer.state_dict(),
            'scheduler_state': self.scheduler.state_dict(),
            'scaler_state': self.scaler.state_dict(),
            'best_val_f1': val_f1,
            'best_val_acc': val_acc
        }
        torch.save(state, self.last_checkpoint_path) # Luôn lưu mốc cuối cùng để Resume
        if is_best: 
            torch.save(state, self.best_checkpoint_path)

    def load_checkpoint(self):
        start_epoch, best_val_f1, best_val_acc = 0, 0.0, 0.0
        if os.path.exists(self.last_checkpoint_path):
            state = torch.load(self.last_checkpoint_path, map_location=device)
            # Dọn dẹp key rác của thop (nếu có)
            clean_state_dict = {k: v for k, v in state['model_state'].items() if 'total_ops' not in k and 'total_params' not in k}
            self.model.load_state_dict(clean_state_dict, strict=False)
            
            if 'optimizer_state' in state:
                self.optimizer.load_state_dict(state['optimizer_state'])
                self.scheduler.load_state_dict(state['scheduler_state'])
                self.scaler.load_state_dict(state['scaler_state'])
                
            start_epoch = state['epoch'] + 1
            best_val_f1 = state.get('best_val_f1', 0.0)
            best_val_acc = state.get('best_val_acc', 0.0)
            print(f"🔋 Đã khôi phục {self.model_name}! Chạy tiếp từ Epoch {start_epoch + 1}...")
        return start_epoch, best_val_f1, best_val_acc

    def log_training(self, row):
        df = pd.read_csv(self.train_log_path)
        df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
        df.to_csv(self.train_log_path, index=False)

def get_backbone_info(model):
    param_count = sum(p.numel() for p in model.backbone.parameters())
    param_memory_mb = sum(p.nelement() * p.element_size() for p in model.backbone.parameters()) / (1024 * 1024)
    return param_count, param_memory_mb

def calculate_routing_metrics(model):
    metrics = {"entropy": 0.0, "expert_usage_distribution": None}
    try:
        moe = model.moe_layer
        if hasattr(moe, "gate_logits") and moe.gate_logits is not None:
            probs = torch.softmax(moe.gate_logits, dim=-1)
            entropy = -(probs * torch.log(probs + 1e-9)).sum(dim=-1).mean()
            metrics["entropy"] = entropy.item()
            metrics["expert_usage_distribution"] = probs.mean(dim=0).cpu().numpy().tolist()
        elif hasattr(moe, "expert_usage") and moe.expert_usage is not None:
            usage = moe.expert_usage.float()
            probs = usage / (usage.sum() + 1e-9)
            entropy = -(probs * torch.log(probs + 1e-9)).sum()
            metrics["entropy"] = entropy.item()
            metrics["expert_usage_distribution"] = usage.cpu().numpy().tolist()
    except Exception: pass
    return metrics


class LayerAttentionPooling(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(hidden_size, hidden_size), 
            nn.Tanh(), 
            nn.Linear(hidden_size, 1)
        )
        
    def forward(self, hidden_states, attention_mask):
        attn_weights = self.attention(hidden_states).squeeze(-1)
        min_val = torch.finfo(attn_weights.dtype).min 
        attn_weights = attn_weights.masked_fill(attention_mask == 0, min_val)
        attn_weights = F.softmax(attn_weights, dim=-1)
        return torch.bmm(attn_weights.unsqueeze(1), hidden_states).squeeze(1)

class UnifiedMoENLI(nn.Module):
    def __init__(self, config, routing_type="expert_choice"):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(config.MODEL_NAME)
        hidden_size = self.backbone.config.hidden_size
        
        # Đóng băng 12 layer đầu
        for name, param in self.backbone.named_parameters():
            if 'encoder.layer' in name and int(name.split('.')[2]) < 12:
                param.requires_grad = False

        # Khởi tạo linh hoạt, loại bỏ hardcode fallback
        if routing_type == "smoe": self.moe_layer = SMoELayer(hidden_size, config.NUM_EXPERTS)
        elif routing_type == "micro": self.moe_layer = MICROMoELayer(hidden_size)
        elif routing_type == "expert_choice": self.moe_layer = ExpertChoiceMoELayer(hidden_size, config.NUM_EXPERTS, config.CAPACITY_FACTOR)
        elif routing_type == "adaptive": self.moe_layer = AdaptiveDynamicMoELayer(hidden_size, config.NUM_EXPERTS, getattr(config, 'ROUTING_THRESHOLDS', 0.5))
        elif routing_type == "deepseek": self.moe_layer = DeepSeekMoELayer(hidden_size, num_shared_experts=2, num_routed_experts=config.NUM_EXPERTS-2)
        else: raise ValueError(f" Routing type '{routing_type}' không hợp lệ!")
            
        self.attention_pooling = LayerAttentionPooling(hidden_size)
        self.classifier = nn.Sequential(
            nn.Dropout(0.2), 
            nn.Linear(hidden_size, hidden_size // 2), 
            nn.GELU(), 
            nn.Linear(hidden_size // 2, config.NUM_LABELS)
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        moe_output = self.moe_layer(outputs.last_hidden_state)
        
        # Xử lý trường hợp MoE layer trả về tuple (output, aux_loss)
        aux_loss = 0.0
        if isinstance(moe_output, tuple):
            moe_output, aux_loss = moe_output
            
        pooled_output = self.attention_pooling(moe_output, attention_mask)
        logits = self.classifier(pooled_output)
        
        return logits, aux_loss

In [5]:
# ==========================================
# CELL CUỐI: MEGA PIPELINE (PHOBERT BASE)
# ==========================================

all_test_results = []

for seed in Config.SEEDS:
    set_seed(seed)
    
    for current_routing in Config.ROUTING_TYPES_TO_TEST:
        print(f"\n{'='*70}\n🚀 SEED {seed} | PHOBERT BASE | ROUTING: {current_routing.upper()} \n{'='*70}")

        is_completed = False
        if os.path.exists(Config.RESULTS_CSV):
            try:
                df_check = pd.read_csv(Config.RESULTS_CSV)
                if not df_check[(df_check['Seed'] == seed) & (df_check['Routing'] == current_routing)].empty:
                    is_completed = True
            except: pass
            
        if is_completed:
            print(f"⏭️ Bỏ qua {current_routing.upper()} vì đã hoàn thành ở lần chạy trước!")
            continue 

        model = UnifiedMoENLI(Config(), routing_type=current_routing).to(device)
        optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=Config.LR, weight_decay=0.01)
        
        acc_steps = getattr(Config, 'ACCUMULATION_STEPS', 1)
        total_steps = math.ceil(len(train_loader) / acc_steps) * Config.EPOCHS
        scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps)
        scaler = GradScaler()
        criterion = nn.CrossEntropyLoss()
        
        model_name = f"phobert_base_{current_routing}_seed{seed}"
        checkpoint_manager = CheckpointManager(model, optimizer, scheduler, scaler, model_name=model_name)
        start_epoch, best_val_f1, best_val_acc = checkpoint_manager.load_checkpoint()

        dummy_ids = torch.ones(1, Config.MAX_LEN, dtype=torch.long).to(device)
        dummy_mask = torch.ones(1, Config.MAX_LEN, dtype=torch.long).to(device)
        macs, _ = profile(model, inputs=(dummy_ids, dummy_mask), verbose=False)
        final_gflops = (macs * 2) / 1e9
        print(f"📊 Tài nguyên ước tính: {final_gflops:.2f} GFlops")

        early_stop_counter = 0

        for epoch in range(start_epoch, Config.EPOCHS):
            model.train()
            optimizer.zero_grad(set_to_none=True) 
            train_iterator = tqdm(train_loader, desc=f"[{current_routing.upper()}] Ep {epoch+1}/{Config.EPOCHS} [Train]", leave=False)
            
            for step, batch in enumerate(train_iterator):
                ids = batch['input_ids'].to(device, non_blocking=True)
                mask = batch['attention_mask'].to(device, non_blocking=True)
                labels = batch['labels'].to(device, non_blocking=True)
                
                with autocast():
                    # LƯU Ý Ở ĐÂY: PhoBERT Base trả về 2 giá trị
                    logits, aux_loss = model(ids, mask)
                    main_loss = criterion(logits, labels)
                    loss = (main_loss + aux_loss) / acc_steps
                    
                scaler.scale(loss).backward()
                
                if (step + 1) % acc_steps == 0 or (step + 1) == len(train_loader):
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    scaler.step(optimizer)
                    scaler.update()
                    scheduler.step()
                    optimizer.zero_grad(set_to_none=True) 
                    
                train_iterator.set_postfix(loss=f"{(loss.item() * acc_steps):.4f}")

            # --- VALIDATION ---
            model.eval()
            val_preds, val_labels = [], []
            torch.cuda.synchronize()
            start_time = time.time()
            
            with torch.inference_mode():
                val_iterator = tqdm(val_loader, desc=f"[{current_routing.upper()}] Ep {epoch+1}/{Config.EPOCHS} [Val]", leave=False)
                for batch in val_iterator:
                    b_ids = batch['input_ids'].to(device, non_blocking=True)
                    b_mask = batch['attention_mask'].to(device, non_blocking=True)
                    b_labels = batch['labels'].to(device, non_blocking=True)
                    
                    with autocast():
                        logits, _ = model(b_ids, b_mask)
                    
                    val_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
                    val_labels.extend(b_labels.cpu().numpy())
                    
            torch.cuda.synchronize()
            runtime_ms = ((time.time() - start_time) / len(val_loader)) * 1000
            vram_mb = torch.cuda.max_memory_allocated() / (1024 * 1024)
            
            val_acc = accuracy_score(val_labels, val_preds)
            val_f1 = f1_score(val_labels, val_preds, average='macro')
            routing_stats = calculate_routing_metrics(model)
            
            print(f"[{current_routing.upper()}] Ep {epoch+1} | Acc: {val_acc:.4f} | F1: {val_f1:.4f} | {runtime_ms:.2f} ms/b | VRAM: {vram_mb:.0f} MB")
            
            checkpoint_manager.log_training({
                "Seed": seed, "Epoch": epoch+1, "Routing": current_routing, 
                "Val_Acc": val_acc, "Val_F1": val_f1, "Val_Runtime_ms": runtime_ms, 
                "Val_VRAM_MB": vram_mb, "Val_Entropy": routing_stats["entropy"], 
                "Val_Expert_Usage": str(routing_stats["expert_usage_distribution"])
            })
            
            is_best = val_f1 > best_val_f1
            if is_best: 
                best_val_f1 = val_f1
                best_val_acc = val_acc
                early_stop_counter = 0 
                print("✨ Validation F1 cải thiện, lưu Best Checkpoint.")
            else:
                early_stop_counter += 1
                print(f"⚠️ Validation F1 không tăng. Early Stop: {early_stop_counter}/{Config.PATIENCE}")
                
            checkpoint_manager.save_checkpoint(epoch, best_val_f1, best_val_acc, is_best)
            
            if early_stop_counter >= Config.PATIENCE:
                print(f"🛑 Kích hoạt Early Stopping tại Epoch {epoch+1}!")
                break

        # --- TEST TẬP CHUẨN ---
        print(f"\n📥 Loading best checkpoint cho {current_routing.upper()} (Seed {seed})...")
        try:
            state = torch.load(checkpoint_manager.best_checkpoint_path, map_location=device)
            clean_state_dict = {k: v for k, v in state["model_state"].items() if 'total_ops' not in k and 'total_params' not in k}
            model.load_state_dict(clean_state_dict, strict=False)
            model.eval()
            
            test_preds, test_labels = [], []
            torch.cuda.reset_peak_memory_stats()
            torch.cuda.synchronize()
            test_start_time = time.time()
            
            with torch.inference_mode():
                for batch in test_loader:
                    ids = batch["input_ids"].to(device, non_blocking=True)
                    mask = batch["attention_mask"].to(device, non_blocking=True)
                    labels = batch["labels"].to(device, non_blocking=True)
                    
                    with autocast():
                        logits, _ = model(ids, mask)
                    test_preds.extend(torch.argmax(logits, 1).cpu().numpy())
                    test_labels.extend(labels.cpu().numpy())

            torch.cuda.synchronize()
            test_runtime_ms = ((time.time() - test_start_time) / len(test_loader)) * 1000
            test_vram_mb = torch.cuda.max_memory_allocated() / (1024 * 1024)

            test_acc = accuracy_score(test_labels, test_preds)
            test_f1 = f1_score(test_labels, test_preds, average="macro")
            test_routing_stats = calculate_routing_metrics(model)
            param_count, param_memory_mb = get_backbone_info(model)

            print(f"🎯 TEST ACC = {test_acc:.4f} | TEST F1 = {test_f1:.4f}")
            
            res_df = pd.DataFrame([{
                "Seed": seed, "Routing": current_routing,
                "Best_Val_Acc": best_val_acc, "Best_Val_F1": best_val_f1,
                "Test_Acc": test_acc, "Test_F1": test_f1, "GFLOPS": final_gflops, 
                "Test_Runtime_ms": test_runtime_ms, "Test_VRAM_MB": test_vram_mb,
                "Test_Entropy": test_routing_stats["entropy"], 
                "Test_Expert_Usage": str(test_routing_stats["expert_usage_distribution"]),
                "Backbone_Params": param_count, "Backbone_Memory_MB": param_memory_mb
            }])
            
            if not os.path.exists(Config.RESULTS_CSV):
                res_df.to_csv(Config.RESULTS_CSV, index=False)
            else:
                res_df.to_csv(Config.RESULTS_CSV, mode='a', header=False, index=False)
                
        except Exception as e:
            print(f"❌ Không thể chạy Test. Lỗi: {e}")
        
        del model, optimizer, scheduler, checkpoint_manager
        torch.cuda.empty_cache()
        gc.collect()

print(f"✅ Hoàn tất huấn luyện! Toàn bộ file đã được lưu tại thư mục: {Config.EXP_DIR}")

 Đã thiết lập Seed = 42

🚀 SEED 42 | PHOBERT BASE | ROUTING: SMOE 

📊 Tài nguyên ước tính: 189.63 GFlops


[SMOE] Ep 1/50 [Train]:   0%|          | 0/501 [00:00<?, ?it/s]

[SMOE] Ep 1/50 [Val]:   0%|          | 0/63 [00:00<?, ?it/s]

[SMOE] Ep 1 | Acc: 0.3330 | F1: 0.1665 | 114.17 ms/b | VRAM: 8687 MB
✨ Validation F1 cải thiện, lưu Best Checkpoint.


[SMOE] Ep 2/50 [Train]:   0%|          | 0/501 [00:00<?, ?it/s]

...

[DEEPSEEK] Ep 6/50 [Val]:   0%|          | 0/63 [00:00<?, ?it/s]

[DEEPSEEK] Ep 6 | Acc: 0.4060 | F1: 0.3984 | 111.81 ms/b | VRAM: 12229 MB
⚠️ Validation F1 không tăng. Early Stop: 2/3


[DEEPSEEK] Ep 7/50 [Train]:   0%|          | 0/501 [00:00<?, ?it/s]

[DEEPSEEK] Ep 7/50 [Val]:   0%|          | 0/63 [00:00<?, ?it/s]

[DEEPSEEK] Ep 7 | Acc: 0.3980 | F1: 0.3949 | 111.30 ms/b | VRAM: 12229 MB
⚠️ Validation F1 không tăng. Early Stop: 3/3
🛑 Kích hoạt Early Stopping tại Epoch 7!

📥 Loading best checkpoint cho DEEPSEEK (Seed 42)...
🎯 TEST ACC = 0.4060 | TEST F1 = 0.3994
✅ Hoàn tất huấn luyện! Toàn bộ file đã được lưu tại thư mục: experiments/PhoBERT\experiment_1
